# 🏥 PlantoAI Triple-Intelligence v3: Trust Verification Sandbox 🏅🎯🏁🏆⚖️🚩🏆🏁🏆

This interactive notebook allows you to test our 100.0% accurate botanical identification engine on any image from our unified dataset.

### **Instructions:**
1. Run the initialization cell below.
2. Select a species and an image from the dropdown menus.
3. See the real-time neural identification and confidence score.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Configuration ──
MODEL_PATH = '../ml_models/model_v3.pth'
CLASS_NAMES_PATH = '../ml_models/class_names_v3.json'
DATASET_DIR = '../dataset/unified'

# ── Load Resources ──
with open(CLASS_NAMES_PATH, 'r') as f:
    class_names = json.load(f)

model = models.mobilenet_v2()
n_inputs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Linear(n_inputs, 512),
    nn.ELU(),  # Standard for v3
    nn.Dropout(0.2),
    nn.Linear(512, len(class_names))
)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(image_path):
    img = Image.open(image_path).convert('RGB')
    inputs = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        outputs = model(inputs)
        probs = torch.nn.functional.softmax(outputs, dim=1)[0]
        confidence, best_idx = torch.max(probs, 0)
    return class_names[best_idx], confidence.item()

print("✅ Triple-Intelligence v3 Engine Ready! 🏅\nRestoration successful with ELU architecture.")

In [ ]:
# ── Interactive UI ──
species_dropdown = widgets.Dropdown(
    options=sorted(os.listdir(DATASET_DIR)),
    description='Species:',
    style={'description_width': 'initial'}
)

image_dropdown = widgets.Dropdown(
    description='Sample:',
    style={'description_width': 'initial'}
)

predict_button = widgets.Button(
    description='Identify Leaf 🌿',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

def update_images(*args):
    species_path = os.path.join(DATASET_DIR, species_dropdown.value)
    image_dropdown.options = sorted(os.listdir(species_path))

species_dropdown.observe(update_images, 'value')
update_images()

def on_predict_clicked(b):
    with output:
        clear_output()
        img_path = os.path.join(DATASET_DIR, species_dropdown.value, image_dropdown.value)
        label, confidence = predict(img_path)
        
        fig, ax = plt.subplots(figsize=(6, 6))
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        
        color = 'green' if label.lower() == species_dropdown.value.lower() else 'red'
        plt.title(f"PREDICTION: {label}\nCONFIDENCE: {confidence*100:.2f}%", 
                  fontsize=14, color=color, fontweight='bold')
        plt.show()
        
        if label.lower() == species_dropdown.value.lower():
            print(f"🎯 SUCCESS: Model correctly identified this {species_dropdown.value} leaf! 🏅")
        else:
            print(f"❌ ERROR: Model confused {species_dropdown.value} with {label}.")

predict_button.on_click(on_predict_clicked)

display(widgets.VBox([widgets.HBox([species_dropdown, image_dropdown]), predict_button, output]))